# IMDB：BERT、DeBERTa-v2-xxlarge 与 LoRA 微调

这是六模型实验之后的独立扩展版。保留相同的“选择模型 → 训练 → 验证 → 导出”流程，预训练模型使用各自的 tokenizer，不再接 GloVe 词索引。

**编写日期：2026-10-11。新增实验尚未在Kaggle执行，没有附加旧结果。** 已用本地随机初始化的小模型检查训练、LoRA、检查点重载和CSV导出；这不代表真实BERT或15亿参数DeBERTa的IMDB成绩，也不代表T4显存已实测。

| MODEL_KEY | 底座 | 默认训练方式 |
| --- | --- | --- |
| `bert_full` | `bert-base-uncased` | 全量微调 |
| `bert_lora` | 同一BERT底座 | query/value上加LoRA，同时训练分类头 |
| `deberta_v2_xxlarge` | `microsoft/deberta-v2-xxlarge`，约1.5B | FP16冻结底座，LoRA及pooler/classifier微调 |

DeBERTa没有换成较小模型。其默认方法也不是全量微调，比较结果时必须写清这个区别。BERT全量和BERT LoRA默认用相同划分、长度、batch和学习率，先做对应对照；之后调整超参数时另建一轮记录。

```mermaid
flowchart LR
 subgraph D[数据处理：产出输入]
  A[原始影评] --> B[去HTML，保留标点]
  B --> C[所选底座的tokenizer]
  C --> I[input_ids与attention_mask]
 end
 subgraph M[模型：产出预测]
  I --> E[预训练编码器与分类头]
  E --> L[交叉熵与梯度更新]
  L --> E
  E --> V[验证accuracy与ROC-AUC]
  V --> S[保存并恢复最佳模型]
  S --> P[预测CSV与概率文件]
 end
```


## 1. 导入后先添加这份比赛数据

在Kaggle用 **Import Notebook** 导入本文件，然后在 **Input → Add Input → Competitions** 搜索并添加 **Bag of Words Meets Bags of Popcorn（word2vec-nlp-tutorial）**；若出现规则页面，先接受比赛规则。导入Notebook不会自动添加数据或开启GPU。

| 必需文件 | 用途 | 应有记录数 |
| --- | --- | --- |
| `labeledTrainData.tsv.zip`，或解压后的`labeledTrainData.tsv` | 带0/1标签的训练影评 | 25,000 |
| `testData.tsv.zip`，或解压后的`testData.tsv` | 需要预测的测试影评 | 25,000 |
| `sampleSubmission.csv` | 提交ID与排列顺序 | 25,000 |

添加整个比赛数据即可，不必手动解压。`unlabeledTrainData.tsv.zip`本次不使用；**不需要GloVe、pickle缓存、单独的Python脚本或Hugging Face登录凭据**。模型和tokenizer由代码从公开模型库下载，所以在设置中开启 **Internet**，选择 **GPU T4 ×2**，然后 **Run All**。

默认选择`bert_full`，完整训练3轮并导出25,000条预测；一次运行所选的一种方法。需要BERT LoRA或DeBERTa时，修改第7节的`MODEL_KEY`后重新运行。本Notebook一次只使用所选的一张卡，默认`cuda:0`，也可改为`cuda:1`；两张T4各16GB，不会自动合并显存。

先运行下面的检查，再安装依赖和下载模型。优先使用官方比赛目录，ZIP直接流式读取；缺文件、多份同名文件、标签异常或测试/模板ID不一致时会立即停止并说明原因。

比赛数据入口：[Data](https://www.kaggle.com/competitions/word2vec-nlp-tutorial/data)。


In [ ]:
"""Kaggle IMDB input discovery and validation; standard library only.

Embedded directly into the notebook before dependency installation/model download.
ZIP members are streamed, never extracted to disk.
"""
from __future__ import annotations

import csv
import io
import zipfile
from contextlib import contextmanager
from dataclasses import dataclass
from pathlib import Path
from typing import Iterator

REQUIRED_FILES = ('labeledTrainData.tsv', 'testData.tsv', 'sampleSubmission.csv')
INPUT_HELP = (
    '在Kaggle的Input面板点击Add Input，选择Competitions，添加'
    'Bag of Words Meets Bags of Popcorn（word2vec-nlp-tutorial）。'
    '需要labeledTrainData.tsv、testData.tsv、sampleSubmission.csv；ZIP形式也支持。'
)


@dataclass(frozen=True)
class InputRef:
    name: str
    path: Path
    member: str | None = None

    @property
    def location(self) -> str:
        return str(self.path) + (f' :: {self.member}' if self.member else '')

    @contextmanager
    def open_text(self) -> Iterator[io.TextIOBase]:
        if self.member is None:
            with self.path.open('r', encoding='utf-8-sig', newline='') as stream:
                yield stream
        else:
            with zipfile.ZipFile(self.path) as archive:
                with archive.open(self.member) as binary:
                    with io.TextIOWrapper(binary, encoding='utf-8-sig', newline='') as stream:
                        yield stream


def _inventory(root: Path) -> tuple[dict[str, list[InputRef]], list[str]]:
    paths = sorted(p for p in root.rglob('*') if p.is_file())
    canonical = {name.casefold(): name for name in REQUIRED_FILES}
    plain = {name: [] for name in REQUIRED_FILES}
    packed = {name: [] for name in REQUIRED_FILES}
    zip_errors = []
    for path in paths:
        name = canonical.get(path.name.casefold())
        if name:
            plain[name].append(InputRef(name, path))
        if path.suffix.lower() != '.zip':
            continue
        try:
            with zipfile.ZipFile(path) as archive:
                for member in archive.namelist():
                    if member.endswith('/') or '__MACOSX' in Path(member).parts:
                        continue
                    name = canonical.get(Path(member).name.casefold())
                    if name:
                        packed[name].append(InputRef(name, path, member))
        except (OSError, zipfile.BadZipFile) as error:
            zip_errors.append(f'{path}: {type(error).__name__}')
    # Prefer an expanded copy over ZIPs from the same folder. A matching ZIP
    # in a different input folder is another source, so keep it as ambiguous.
    selected = {}
    for name in REQUIRED_FILES:
        plain_folders = {ref.path.parent for ref in plain[name]}
        selected[name] = plain[name] + [
            ref for ref in packed[name] if ref.path.parent not in plain_folders
        ]
    return selected, zip_errors


def discover_inputs(input_root: str | Path = '/kaggle/input',
                    data_dir: str | Path | None = None) -> dict[str, InputRef]:
    root = Path(data_dir) if data_dir is not None else Path(input_root)
    if not root.is_dir():
        raise FileNotFoundError(f'输入目录不存在：{root}。{INPUT_HELP}')
    preferred = [] if data_dir is not None else [
        root / 'competitions' / 'word2vec-nlp-tutorial',
        root / 'word2vec-nlp-tutorial',
    ]
    selected = None
    for folder in preferred:
        if folder.is_dir():
            selected, errors = _inventory(folder)
            root = folder
            break
    if selected is None:
        selected, errors = _inventory(root)
    missing = [name for name, choices in selected.items() if not choices]
    ambiguous = {name: [choice.location for choice in choices]
                 for name, choices in selected.items() if len(choices) > 1}
    if missing:
        detail = f'；发现损坏ZIP：{errors}' if errors else ''
        raise FileNotFoundError(f'目录{root}缺少：{missing}{detail}。{INPUT_HELP}')
    if ambiguous:
        raise ValueError(f'发现多份同名数据：{ambiguous}。请把DATA_DIR设置为要使用的具体比赛目录。')
    return {name: choices[0] for name, choices in selected.items()}


def _id(value: str | None, name: str, row: int) -> str:
    result = (value or '').strip().strip('"').strip()
    if not result:
        raise ValueError(f'{name}第{row}条记录的id为空。')
    return result


def validate_inputs(files: dict[str, InputRef], expected_rows: int = 25000) -> dict:
    csv.field_size_limit(10_000_000)
    report = {'paths': {}, 'rows': {}, 'label_counts': {'0': 0, '1': 0}}
    ids = {}
    for name in REQUIRED_FILES:
        if name not in files:
            raise FileNotFoundError(f'未提供{name}。{INPUT_HELP}')
        ref = files[name]
        is_sample = name.endswith('.csv')
        required = {'id', 'sentiment'} if is_sample else {'id', 'review'}
        if name == 'labeledTrainData.tsv':
            required.add('sentiment')
        values = []
        with ref.open_text() as stream:
            reader = csv.DictReader(stream, delimiter=',' if is_sample else '\t',
                                    quoting=csv.QUOTE_MINIMAL if is_sample else csv.QUOTE_NONE)
            if not required.issubset(set(reader.fieldnames or [])):
                raise ValueError(f'{name}列名错误：需要{sorted(required)}，实际{reader.fieldnames}。')
            for row_number, row in enumerate(reader, 1):
                if None in row or any(row.get(column) is None for column in required):
                    raise ValueError(f'{name}第{row_number}条记录的列数不一致。')
                values.append(_id(row.get('id'), name, row_number))
                if not is_sample and not row['review'].strip().strip('"').strip():
                    raise ValueError(f'{name}第{row_number}条记录的review为空。')
                if name == 'labeledTrainData.tsv':
                    label = row['sentiment'].strip()
                    if label not in {'0', '1'}:
                        raise ValueError(f'{name}第{row_number}条记录的标签须为0或1，实际为{label!r}。')
                    report['label_counts'][label] += 1
        if len(values) != expected_rows:
            raise ValueError(f'{name}应有{expected_rows}条记录，实际{len(values)}条。')
        if len(set(values)) != len(values):
            raise ValueError(f'{name}存在重复id。')
        ids[name] = values
        report['paths'][name] = ref.location
        report['rows'][name] = len(values)
    if set(ids['testData.tsv']) != set(ids['sampleSubmission.csv']):
        raise ValueError('testData.tsv与sampleSubmission.csv的id集合不一致；请添加同一比赛的完整数据。')
    if not all(report['label_counts'].values()):
        raise ValueError('训练标签必须同时包含0和1两类。')
    report['test_order_matches_template'] = ids['testData.tsv'] == ids['sampleSubmission.csv']
    report['status'] = 'ready'
    return report


def read_dataframe(ref: InputRef):
    import pandas as pd
    with ref.open_text() as stream:
        return pd.read_csv(stream, sep=',' if ref.name.endswith('.csv') else '\t',
                           quoting=csv.QUOTE_MINIMAL if ref.name.endswith('.csv') else csv.QUOTE_NONE,
                           dtype={'id': str})


In [ ]:
DATA_DIR = None  # 自动定位；有自定义目录时填"/kaggle/input/具体目录"，不要填ZIP文件名。
INPUT_FILES = discover_inputs(data_dir=DATA_DIR)
INPUT_REPORT = validate_inputs(INPUT_FILES)
for name in REQUIRED_FILES:
    print(f"{name}: {INPUT_REPORT['rows'][name]:,}条")
    print("  ", INPUT_REPORT["paths"][name])
print("训练标签分布:", INPUT_REPORT["label_counts"])
print("测试顺序与模板一致:", INPUT_REPORT["test_order_matches_template"])
print("输入检查通过。可以继续安装依赖并训练。")


## 2. 安装依赖并检查GPU

以下安装固定Transformers/PEFT版本，保留Kaggle已有CUDA版PyTorch。安装后如果提示已导入的库版本冲突，重启会话后从头Run All，以重新建立数据检查与共用函数。真实checkpoint读取要求PyTorch≥2.6，不要把本地测试的CPU版torch装到Kaggle。


In [ ]:
%pip install -q "transformers==5.19.0" "peft==0.21.2" "accelerate==1.15.0" "sentencepiece==0.2.2" "protobuf>=4.25,<8" "beautifulsoup4==4.15.0" "pandas>=2.2,<4" "scikit-learn>=1.5,<2"


In [ ]:
import sys
import importlib.metadata
import torch
from packaging.version import Version

print("Python:", sys.version.split()[0])
print("torch:", torch.__version__)
if Version(torch.__version__.split("+")[0]) < Version("2.6"):
    raise RuntimeError("需要PyTorch>=2.6；请保留/安装适配Kaggle CUDA环境的版本后重启会话。")
for name in ("transformers", "peft", "accelerate"):
    print(name, importlib.metadata.version(name))
if not torch.cuda.is_available():
    raise RuntimeError("请先在Kaggle设置里打开GPU；本地只做小模型测试。")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}: {p.name}, {p.total_memory / 2**30:.2f} GiB")


## 3. 共用配置与数据接口

下面三个代码单元来自同目录的 `code/imdb_pretrained.py`，完整嵌入，导入Kaggle后不用再下载脚本。通常只需要修改后面的模型配置单元。

不同底座的词表和分词方式不同，所以模型与tokenizer成对切换。LoRA更新的是权重的低秩增量，不是把影评输入直接降到8维，也不是写8条规则。

$$W=W_0+\frac{\alpha}{r}BA,\qquad A\in\mathbb{R}^{r\times k},\quad B\in\mathbb{R}^{d\times r}.$$

`W_0`冻结；`A`、`B`与任务分类头参与训练。DeBERTa的pooler也是新增任务层，必须一起训练和保存。


In [ ]:
"""One-model-at-a-time IMDB fine-tuning, separate from the historical GloVe notebook.

Required packages: torch, transformers, scikit-learn; peft for LoRA;
sentencepiece for DeBERTa-v2.
Version pins belong in the tested environment file, not in this reusable module.

Example::
    cfg = TrainingConfig(output_dir="result/bert_lora", num_epochs=3)
    run = train_model("bert_lora", train_frame, validation_frame, config=cfg)
    predict_submit(run, test_frame, "result/bert_lora/submission.csv")

The registry uses the actual 1.5B DeBERTa-v2-xxlarge, not a smaller replacement.
Its default is LoRA plus pooler/classifier training. Each T4 is an independent
16 GiB device: cuda:0 and cuda:1 are selectable, never implicitly combined.
Full XXL tuning requires explicit opt-in and a conservative free-memory check.
Registry/source overrides and injected tiny models support offline smoke tests.
Injected adapter/head-only models need a matching local model_name_or_path for
automatic reload, or an explicitly supplied base_model in load_run().
"""

from __future__ import annotations

import csv
import hashlib
import importlib.metadata
import json
import math
import os
import random
import re
from dataclasses import asdict, dataclass, replace
from datetime import datetime, timezone
from html.parser import HTMLParser
from itertools import islice
from pathlib import Path
from typing import Any, Mapping, Optional, Sequence

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset


@dataclass(frozen=True)
class ModelSpec:
    model_name_or_path: str
    tokenizer_name_or_path: str
    default_batch_size: int
    default_eval_batch_size: int
    default_max_length: int
    default_accumulation_steps: int
    lora_targets: tuple[str, ...] = ()
    modules_to_save: tuple[str, ...] = ("classifier",)


MODEL_REGISTRY: dict[str, ModelSpec] = {
    "bert_full": ModelSpec("bert-base-uncased", "bert-base-uncased", 8, 16, 256, 2),
    "bert_lora": ModelSpec(
        "bert-base-uncased", "bert-base-uncased", 8, 16, 256, 2, ("query", "value")
    ),
    "deberta_v2_xxlarge": ModelSpec(
        "microsoft/deberta-v2-xxlarge", "microsoft/deberta-v2-xxlarge",
        1, 1, 128, 16, ("query_proj", "value_proj"), ("classifier", "pooler"),
    ),
}


@dataclass
class TrainingConfig:
    model_key: Optional[str] = None
    output_dir: str = "result/pretrained_imdb"
    num_epochs: int = 3
    batch_size: Optional[int] = None
    eval_batch_size: Optional[int] = None
    max_length: Optional[int] = None
    gradient_accumulation_steps: Optional[int] = None
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    max_grad_norm: Optional[float] = 1.0
    amp: bool = True
    seed: int = 42
    deterministic: bool = True
    device: Optional[str] = None
    num_workers: int = 0
    review_column: str = "review"
    label_column: str = "sentiment"
    xxl_mode: str = "lora"  # lora, head_only, or full
    allow_xxl_full: bool = False
    xxl_min_free_gib: float = 40.0
    gradient_checkpointing: Optional[bool] = None
    lora_r: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.05
    model_name_or_path: Optional[str] = None
    tokenizer_name_or_path: Optional[str] = None
    revision: Optional[str] = None
    tokenizer_revision: Optional[str] = None
    local_files_only: bool = False
    monitor: str = "accuracy"  # accuracy, auc, or loss; AUC matches the Kaggle metric


@dataclass
class TrainingRun:
    model: nn.Module
    tokenizer: Any
    config: TrainingConfig
    history: list[dict[str, Any]]
    best_epoch: int
    best_validation: dict[str, Any]
    output_dir: Path
    metadata: dict[str, Any]


@dataclass
class PredictionResult:
    submission_path: Path
    probability_path: Path
    n_examples: int
    ids: list[str]
    labels: list[int]
    positive_probabilities: list[float]


def _resolve_config(model_key: str, config: Optional[TrainingConfig]) -> TrainingConfig:
    if model_key not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model_key {model_key!r}; choose {tuple(MODEL_REGISTRY)}")
    cfg = replace(config or TrainingConfig())
    if cfg.model_key is not None and cfg.model_key != model_key:
        raise ValueError("config.model_key and train_model model_key disagree")
    spec = MODEL_REGISTRY[model_key]
    cfg.model_key = model_key
    cfg.batch_size = cfg.batch_size if cfg.batch_size is not None else spec.default_batch_size
    cfg.eval_batch_size = (
        cfg.eval_batch_size if cfg.eval_batch_size is not None else spec.default_eval_batch_size
    )
    cfg.max_length = cfg.max_length if cfg.max_length is not None else spec.default_max_length
    cfg.gradient_accumulation_steps = (
        cfg.gradient_accumulation_steps
        if cfg.gradient_accumulation_steps is not None else spec.default_accumulation_steps
    )
    if cfg.gradient_checkpointing is None:
        cfg.gradient_checkpointing = model_key == "deberta_v2_xxlarge" and cfg.xxl_mode != "head_only"
    for field in ("num_epochs", "batch_size", "eval_batch_size", "max_length", "gradient_accumulation_steps", "lora_r"):
        value = getattr(cfg, field)
        if not isinstance(value, int) or value < 1:
            raise ValueError(f"{field} must be a positive integer")
    if cfg.num_workers < 0 or cfg.learning_rate <= 0 or cfg.weight_decay < 0:
        raise ValueError("Invalid workers, learning rate, or weight decay")
    if cfg.xxl_mode not in {"lora", "head_only", "full"}:
        raise ValueError("xxl_mode must be lora, head_only, or full")
    if not 0 <= cfg.lora_dropout < 1 or cfg.lora_alpha <= 0:
        raise ValueError("Invalid LoRA alpha/dropout")
    if cfg.monitor not in {"accuracy", "auc", "loss"}:
        raise ValueError("monitor must be accuracy, auc, or loss")
    if cfg.max_grad_norm is not None and cfg.max_grad_norm <= 0:
        raise ValueError("max_grad_norm must be positive or None")
    if cfg.xxl_min_free_gib < 30:
        raise ValueError("XXL full memory guard must allow at least 30 GiB before activations")
    cfg.output_dir = str(Path(cfg.output_dir).resolve())
    return cfg


def set_reproducibility(seed: int, deterministic: bool = True) -> None:
    """Seed training and loader shuffle; deterministic CUDA behavior is best-effort."""
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except ImportError:
        pass
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = not deterministic
    torch.backends.cudnn.deterministic = deterministic
    torch.use_deterministic_algorithms(deterministic, warn_only=True)


def _device(device: Optional[str]) -> torch.device:
    result = torch.device(device or ("cuda:0" if torch.cuda.is_available() else "cpu"))
    if result.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but unavailable")
    if result.type not in {"cpu", "cuda"}:
        raise ValueError("This training interface supports cpu or a single cuda device")
    return result


def _guard_xxl_full(cfg: TrainingConfig, device: torch.device, injected: bool,
                    parameter_count: Optional[int] = None) -> None:
    if cfg.model_key != "deberta_v2_xxlarge" or cfg.xxl_mode != "full":
        return
    if not cfg.allow_xxl_full:
        raise ValueError("XXL full tuning needs allow_xxl_full=True; default XXL mode is lora")
    source = cfg.model_name_or_path or MODEL_REGISTRY[cfg.model_key].model_name_or_path
    # Explicit tiny local overrides/injections are for tests, not an alternate XXL model.
    if parameter_count is not None and parameter_count < 1_000_000_000:
        return
    if parameter_count is None and (injected or source != MODEL_REGISTRY[cfg.model_key].model_name_or_path):
        return
    if device.type != "cuda":
        raise RuntimeError("Real 1.5B XXL full tuning requires a suitable CUDA device")
    free_bytes, _ = torch.cuda.mem_get_info(device)
    free_gib = free_bytes / 2**30
    if free_gib < cfg.xxl_min_free_gib:
        raise RuntimeError(
            f"XXL full Adam needs about 27 GB before activations. {device} has "
            f"{free_gib:.1f} GiB free; this run requires {cfg.xxl_min_free_gib:.1f}. "
            "Two 16 GiB T4s do not form one 32 GiB device. Choose lora/head_only."
        )


## 4. tokenizer、模型构建与验证

动态padding只补到当前batch的最长序列；`attention_mask`告诉模型哪些位置有效。验证准确率按样本总数统计，ROC-AUC用正类概率统计。


In [ ]:
class _HTMLText(HTMLParser):
    def __init__(self) -> None:
        super().__init__(convert_charrefs=True)
        self.parts: list[str] = []

    def handle_data(self, data: str) -> None:
        self.parts.append(data)

    def handle_starttag(self, tag: str, attrs: Any) -> None:
        self.parts.append(" ")

    def handle_endtag(self, tag: str) -> None:
        self.parts.append(" ")


def strip_html(review: str) -> str:
    """Preserve punctuation/case for the selected model's own tokenizer."""
    if not isinstance(review, str):
        raise ValueError("Each review must be a non-missing string")
    parser = _HTMLText()
    parser.feed(review)
    parser.close()
    return re.sub(r"\s+", " ", "".join(parser.parts)).strip()


def _records(frame: Any) -> list[dict[str, Any]]:
    if isinstance(frame, (str, Path)):
        with Path(frame).open(encoding="utf-8-sig", newline="") as handle:
            return list(csv.DictReader(handle))
    if hasattr(frame, "to_dict"):
        return frame.to_dict(orient="records")
    return [dict(row) for row in frame]


class ReviewDataset(Dataset):
    def __init__(self, frame: Any, review_column: str = "review", label_column: Optional[str] = "sentiment"):
        rows = _records(frame)
        if not rows:
            raise ValueError("Dataset is empty")
        self.texts: list[str] = []
        self.labels: Optional[list[int]] = [] if label_column is not None else None
        for row in rows:
            if review_column not in row:
                raise ValueError(f"Missing review column {review_column!r}")
            self.texts.append(strip_html(row[review_column]))
            if label_column is not None:
                value = row.get(label_column)
                try:
                    label = int(value)
                    valid = label in (0, 1) and float(value) == label
                except (ValueError, TypeError, OverflowError):
                    valid = False
                if not valid:
                    raise ValueError("Training labels must be 0 or 1")
                self.labels.append(label)

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, index: int) -> dict[str, Any]:
        item = {"text": self.texts[index]}
        if self.labels is not None:
            item["labels"] = self.labels[index]
        return item

    def fingerprint(self) -> str:
        payload = json.dumps([self.texts, self.labels], ensure_ascii=False, separators=(",", ":"))
        return hashlib.sha256(payload.encode("utf-8")).hexdigest()


@dataclass
class ReviewCollator:
    tokenizer: Any
    max_length: int

    def __call__(self, examples: Sequence[Mapping[str, Any]]) -> dict[str, torch.Tensor]:
        encoded = self.tokenizer(
            [item["text"] for item in examples], truncation=True,
            padding=True, max_length=self.max_length, return_tensors="pt",
        )
        batch = {key: value for key, value in encoded.items()
                 if key in {"input_ids", "attention_mask", "token_type_ids"}}
        if "input_ids" not in batch or "attention_mask" not in batch:
            raise ValueError("Tokenizer must return input_ids and attention_mask")
        if "labels" in examples[0]:
            batch["labels"] = torch.tensor([item["labels"] for item in examples], dtype=torch.long)
        return batch


def _seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % 2**32
    random.seed(worker_seed)
    try:
        import numpy as np
        np.random.seed(worker_seed)
    except ImportError:
        pass


def make_data_loader(dataset: ReviewDataset, tokenizer: Any, cfg: TrainingConfig, *, training: bool) -> DataLoader:
    cfg = _resolve_config(cfg.model_key or "bert_full", cfg)
    generator = torch.Generator().manual_seed(cfg.seed)
    return DataLoader(
        dataset, batch_size=cfg.batch_size if training else cfg.eval_batch_size,
        shuffle=training, drop_last=False, num_workers=cfg.num_workers,
        collate_fn=ReviewCollator(tokenizer, cfg.max_length),
        generator=generator, worker_init_fn=_seed_worker,
    )


def _guard_bin_checkpoint_loader(model_key: str, source: str) -> None:
    # The official XXL repository supplies a PyTorch .bin checkpoint. Current
    # Transformers requires Torch >=2.6 for its safe weights-only loading path.
    if model_key != "deberta_v2_xxlarge" or source != MODEL_REGISTRY[model_key].model_name_or_path:
        return
    version = re.match(r"(\d+)\.(\d+)", torch.__version__)
    if version is None or tuple(map(int, version.groups())) < (2, 6):
        raise RuntimeError("Loading the official DeBERTa-v2 XXL .bin checkpoint requires torch>=2.6")


def build_model_and_tokenizer(
    model_key: str, config: Optional[TrainingConfig] = None, *,
    model: Optional[nn.Module] = None, tokenizer: Any = None,
) -> tuple[nn.Module, Any]:
    """Build one registry choice. Source overrides must match the injected model."""
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    cfg = _resolve_config(model_key, config)
    set_reproducibility(cfg.seed, cfg.deterministic)
    spec = MODEL_REGISTRY[model_key]
    device = _device(cfg.device)
    injected = model is not None
    _guard_xxl_full(cfg, device, injected)
    source = cfg.model_name_or_path or spec.model_name_or_path
    token_source = cfg.tokenizer_name_or_path or cfg.model_name_or_path or spec.tokenizer_name_or_path
    if tokenizer is None:
        token_revision = cfg.tokenizer_revision or (cfg.revision if token_source == source else None)
        tokenizer = AutoTokenizer.from_pretrained(
            token_source, revision=token_revision, local_files_only=cfg.local_files_only,
        )
    if model is None:
        _guard_bin_checkpoint_loader(model_key, source)
        dtype = torch.float16 if (
            model_key == "deberta_v2_xxlarge" and cfg.xxl_mode != "full"
            and device.type == "cuda" and cfg.amp
        ) else torch.float32
        model = AutoModelForSequenceClassification.from_pretrained(
            source, num_labels=2, torch_dtype=dtype, revision=cfg.revision,
            local_files_only=cfg.local_files_only,
        )
    if getattr(model.config, "num_labels", 2) != 2:
        raise ValueError("The model must have exactly two classification outputs")
    _guard_xxl_full(cfg, device, injected,
                    parameter_count=sum(parameter.numel() for parameter in model.parameters()))
    use_lora = model_key == "bert_lora" or (model_key == "deberta_v2_xxlarge" and cfg.xxl_mode == "lora")
    if use_lora:
        from peft import LoraConfig, PeftModel, TaskType, get_peft_model
        if not isinstance(model, PeftModel):
            module_names = {name.rsplit(".", 1)[-1] for name, _ in model.named_modules()}
            missing = set(spec.lora_targets) - module_names
            if missing:
                raise ValueError(f"Model does not have requested LoRA targets: {sorted(missing)}")
            model = get_peft_model(model, LoraConfig(
                task_type=TaskType.SEQ_CLS, r=cfg.lora_r, lora_alpha=cfg.lora_alpha,
                lora_dropout=cfg.lora_dropout, target_modules=list(spec.lora_targets),
                modules_to_save=list(spec.modules_to_save), bias="none",
            ))
        scope = "lora_adapters_classifier_pooler" if model_key == "deberta_v2_xxlarge" else "lora_adapters_classifier"
    elif model_key == "deberta_v2_xxlarge" and cfg.xxl_mode == "head_only":
        for name, parameter in model.named_parameters():
            parameter.requires_grad = name.split(".", 1)[0] in {"classifier", "pooler", "score", "pre_classifier"}
        scope = "head_only_classifier_pooler"
    else:
        for parameter in model.parameters():
            parameter.requires_grad = True
        scope = "full_fine_tuning"
    # GradScaler cannot unscale FP16 parameter gradients. Keep the small trainable
    # adapters/task heads in FP32 while a frozen XXL CUDA backbone can stay FP16.
    for parameter in model.parameters():
        if parameter.requires_grad and parameter.is_floating_point() and parameter.dtype == torch.float16:
            parameter.data = parameter.data.float()
    if not any(parameter.requires_grad for parameter in model.parameters()):
        raise ValueError("No trainable parameters; check task-head or adapter configuration")
    if cfg.gradient_checkpointing:
        if hasattr(model, "gradient_checkpointing_enable"):
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        if use_lora and hasattr(model, "enable_input_require_grads"):
            model.enable_input_require_grads()
        if hasattr(model.config, "use_cache"):
            model.config.use_cache = False
    actual_source = cfg.model_name_or_path or (
        getattr(model.config, "_name_or_path", None) if injected else source
    )
    model._imdb_runtime = {
        "training_scope": scope, "checkpoint_kind": "adapter" if use_lora else ("head_only" if scope.startswith("head_only") else "full"),
        "base_model_source": actual_source or None, "injected_model": injected,
        "source_override": source != spec.model_name_or_path,
        "tokenizer_source": token_source, "lora_targets": list(spec.lora_targets) if use_lora else [],
        "modules_to_save": list(spec.modules_to_save) if use_lora else [],
    }
    return model, tokenizer


def _autocast(device: torch.device, amp: bool):
    return torch.autocast(device_type=device.type, dtype=torch.float16 if device.type == "cuda" else torch.bfloat16,
                          enabled=bool(amp and device.type == "cuda"))


def _logits(model: nn.Module, batch: Mapping[str, torch.Tensor], device: torch.device) -> torch.Tensor:
    inputs = {key: value.to(device) for key, value in batch.items() if key != "labels"}
    output = model(**inputs)
    logits = output.logits
    if logits.ndim != 2 or logits.shape[1] != 2 or not torch.isfinite(logits).all():
        raise ValueError("Expected finite [batch, 2] logits")
    return logits


@torch.no_grad()
def evaluate(model: nn.Module, data_loader: DataLoader, device: Optional[str] = None, *, amp: bool = False) -> dict[str, Any]:
    target = _device(device or str(next(model.parameters()).device))
    model.eval()
    loss_sum, correct, count = 0.0, 0, 0
    all_labels, positive = [], []
    for batch in data_loader:
        labels = batch["labels"].to(target)
        with _autocast(target, amp):
            logits = _logits(model, batch, target)
            loss = F.cross_entropy(logits.float(), labels, reduction="sum")
        loss_sum += loss.item()
        correct += (logits.argmax(dim=-1) == labels).sum().item()
        count += labels.numel()
        all_labels.extend(labels.cpu().tolist())
        positive.extend(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().tolist())
    if count == 0:
        raise ValueError("Validation loader is empty")
    if len(set(all_labels)) == 2:
        from sklearn.metrics import roc_auc_score
        auc = float(roc_auc_score(all_labels, positive))
    else:
        auc = None
    return {"loss": loss_sum / count, "accuracy": correct / count, "auc": auc, "n_examples": count}


def _write_json(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")


def _write_csv(path: Path, rows: Sequence[Mapping[str, Any]], fieldnames: Sequence[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def _versions() -> dict[str, Optional[str]]:
    result: dict[str, Optional[str]] = {}
    for package in ("torch", "transformers", "peft", "accelerate", "sentencepiece", "scikit-learn"):
        try:
            result[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            result[package] = None
    return result


def _save_checkpoint(model: nn.Module, tokenizer: Any, path: Path, kind: str) -> None:
    path.mkdir(parents=True, exist_ok=True)
    if kind == "head_only":
        model.config.save_pretrained(path)
        state = {name: parameter.detach().cpu() for name, parameter in model.named_parameters() if parameter.requires_grad}
        torch.save(state, path / "task_head.pt")
    else:
        # PEFT saves adapters AND modules_to_save (including the random task head).
        model.save_pretrained(path)
    tokenizer.save_pretrained(path)


## 5. 共用训练与预测

梯度累积按每组真实样本数归一化，最后不完整的一组也会更新。每轮验证后选最佳检查点，训练结束恢复该轮参数，再导出预测。LoRA检查点保存adapter和任务层，重新使用时仍需同一个预训练底座。


In [ ]:
def train_model(
    model_key: str, train_frame: Any, validation_frame: Any, *,
    config: Optional[TrainingConfig] = None, model: Optional[nn.Module] = None,
    tokenizer: Any = None,
) -> TrainingRun:
    """Train ONE model, export history/config, and return the restored best model.

    Accumulation groups divide SUM losses by the group's actual example count,
    including short final batches and the incomplete final accumulation group.
    """
    cfg = _resolve_config(model_key, config)
    train_data = ReviewDataset(train_frame, cfg.review_column, cfg.label_column)
    validation_data = ReviewDataset(validation_frame, cfg.review_column, cfg.label_column)
    output_dir = Path(cfg.output_dir)
    if (output_dir / "training_metadata.json").exists():
        raise FileExistsError("This output_dir already has a run; choose a new directory")
    set_reproducibility(cfg.seed, cfg.deterministic)
    target = _device(cfg.device)
    cfg.device = str(target)
    model, tokenizer = build_model_and_tokenizer(model_key, cfg, model=model, tokenizer=tokenizer)
    model.to(target)
    parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    optimizer = torch.optim.AdamW(parameters, lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    amp_enabled = bool(cfg.amp and target.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    train_loader = make_data_loader(train_data, tokenizer, cfg, training=True)
    validation_loader = make_data_loader(validation_data, tokenizer, cfg, training=False)
    runtime = dict(model._imdb_runtime)
    metadata = {
        "schema_version": 1, "status": "running", "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "config": asdict(cfg), "registry_model_source": MODEL_REGISTRY[model_key].model_name_or_path,
        **runtime, "base_revision": cfg.revision,
        "total_parameters": sum(parameter.numel() for parameter in model.parameters()),
        "trainable_parameters": sum(parameter.numel() for parameter in parameters),
        "train_examples": len(train_data), "validation_examples": len(validation_data),
        "train_fingerprint": train_data.fingerprint(), "validation_fingerprint": validation_data.fingerprint(),
        "metric_aggregation": "sample_weighted", "amp_enabled": amp_enabled,
        "gpu_memory_policy": "one selected CUDA device, never pooled across GPUs",
        "device_name": torch.cuda.get_device_name(target) if target.type == "cuda" else "CPU",
        "versions": _versions(), "checkpoint_directory": "best_checkpoint",
    }
    _write_json(output_dir / "training_metadata.json", metadata)
    history: list[dict[str, Any]] = []
    best_epoch, best_validation, best_state = 0, None, None
    total_steps, skipped_steps = 0, 0
    for epoch in range(1, cfg.num_epochs + 1):
        model.train()
        loss_sum, correct, count, epoch_steps = 0.0, 0, 0, 0
        batch_iterator = iter(train_loader)
        while True:
            group = list(islice(batch_iterator, cfg.gradient_accumulation_steps))
            if not group:
                break
            group_count = sum(batch["labels"].numel() for batch in group)
            optimizer.zero_grad(set_to_none=True)
            for batch in group:
                labels = batch["labels"].to(target)
                with _autocast(target, amp_enabled):
                    logits = _logits(model, batch, target)
                    summed_loss = F.cross_entropy(logits.float(), labels, reduction="sum")
                    normalized_loss = summed_loss / group_count
                if not torch.isfinite(summed_loss):
                    raise FloatingPointError("Non-finite training loss")
                scaler.scale(normalized_loss).backward()
                loss_sum += summed_loss.detach().item()
                correct += (logits.detach().argmax(dim=-1) == labels).sum().item()
                count += labels.numel()
            scaler.unscale_(optimizer)
            if cfg.max_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(parameters, cfg.max_grad_norm)
            previous_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= previous_scale:
                epoch_steps += 1
                total_steps += 1
            else:
                skipped_steps += 1
        validation = evaluate(model, validation_loader, str(target), amp=amp_enabled)
        if validation.get(cfg.monitor) is None:
            raise ValueError("AUC selection requires both classes in validation; use monitor='accuracy' or 'loss'")
        row = {
            "epoch": epoch, "train_loss": loss_sum / count, "train_accuracy": correct / count,
            "validation_loss": validation["loss"], "validation_accuracy": validation["accuracy"],
            "validation_auc": validation.get("auc"),
            "train_examples": count, "validation_examples": validation["n_examples"],
            "optimizer_steps": epoch_steps,
        }
        history.append(row)
        better = best_validation is None or (
            validation[cfg.monitor] < best_validation[cfg.monitor] if cfg.monitor == "loss"
            else validation[cfg.monitor] > best_validation[cfg.monitor]
        )
        tied_lower_loss = best_validation is not None and (
            validation[cfg.monitor] == best_validation[cfg.monitor]
            and validation["loss"] < best_validation["loss"]
        )
        if better or tied_lower_loss:
            best_epoch, best_validation = epoch, dict(validation)
            best_state = {name: parameter.detach().cpu().clone()
                          for name, parameter in model.named_parameters() if parameter.requires_grad}
            _save_checkpoint(model, tokenizer, output_dir / "best_checkpoint", runtime["checkpoint_kind"])
        _write_csv(output_dir / "training_history.csv", history, list(row))
        print(f"Epoch {epoch}/{cfg.num_epochs}: Train Loss {row['train_loss']:.4f} | "
              f"Train Acc {row['train_accuracy']:.4f} | Val Acc {row['validation_accuracy']:.4f}")
    # Only trainable parameters changed; restore them in-place, without loading a
    # second 1.5B backbone into GPU memory. No running-stat buffers in these models.
    with torch.no_grad():
        for name, parameter in model.named_parameters():
            if name in best_state:
                parameter.copy_(best_state[name].to(parameter.device))
    model.eval()
    metadata.update({
        "status": "completed", "finished_at_utc": datetime.now(timezone.utc).isoformat(),
        "best_epoch": best_epoch, "best_validation": best_validation,
        "optimizer_steps": total_steps, "skipped_amp_steps": skipped_steps,
        "selection_rule": ("minimum" if cfg.monitor == "loss" else "maximum") +
                          f" validation {cfg.monitor}; ties use lower validation loss",
        "returned_model": "restored best validation checkpoint",
    })
    _write_json(output_dir / "training_metadata.json", metadata)
    _write_json(output_dir / "results.json", {
        "model_key": model_key, "training_scope": runtime["training_scope"],
        "best_epoch": best_epoch, "best_validation": best_validation,
        "final_epoch_validation": {"loss": history[-1]["validation_loss"], "accuracy": history[-1]["validation_accuracy"],
                                   "auc": history[-1]["validation_auc"]},
        "metric_aggregation": "sample_weighted", "kaggle_leaderboard_score": None,
    })
    return TrainingRun(model, tokenizer, cfg, history, best_epoch, best_validation, output_dir, metadata)


def load_run(output_dir: str | Path, *, device: Optional[str] = None,
             base_model: Optional[nn.Module] = None, tokenizer: Any = None) -> TrainingRun:
    """Reload the best checkpoint. Adapter/head-only runs share their original base."""
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    root = Path(output_dir).resolve()
    metadata = json.loads((root / "training_metadata.json").read_text(encoding="utf-8"))
    if metadata.get("status") != "completed":
        raise ValueError("Run did not complete; its checkpoint has no completed-run contract")
    cfg = TrainingConfig(**metadata["config"])
    target = _device(device or cfg.device)
    cfg.device = str(target)
    checkpoint = root / metadata["checkpoint_directory"]
    if tokenizer is None:
        tokenizer = AutoTokenizer.from_pretrained(checkpoint, local_files_only=True)
    kind = metadata["checkpoint_kind"]
    dtype = torch.float16 if (cfg.model_key == "deberta_v2_xxlarge" and kind != "full" and target.type == "cuda" and cfg.amp) else torch.float32
    if kind == "full":
        model = AutoModelForSequenceClassification.from_pretrained(checkpoint, torch_dtype=dtype, local_files_only=True)
    else:
        source = metadata.get("base_model_source")
        if base_model is None:
            if not source:
                raise ValueError("Injected run has no base source; supply its matching base_model")
            _guard_bin_checkpoint_loader(cfg.model_key, source)
            base_model = AutoModelForSequenceClassification.from_pretrained(
                source, num_labels=2, torch_dtype=dtype,
                revision=metadata.get("base_revision"), local_files_only=cfg.local_files_only,
            )
        if kind == "adapter":
            from peft import PeftModel
            for name, parameter in base_model.named_parameters():
                if name.split(".", 1)[0] in {"classifier", "pooler", "score", "pre_classifier"}:
                    parameter.data = parameter.data.float()
            model = PeftModel.from_pretrained(base_model, checkpoint, is_trainable=False)
        else:
            model = base_model
            for name, parameter in model.named_parameters():
                is_head = name.split(".", 1)[0] in {"classifier", "pooler", "score", "pre_classifier"}
                parameter.requires_grad = is_head
                if is_head:
                    parameter.data = parameter.data.float()
            state = torch.load(checkpoint / "task_head.pt", map_location="cpu", weights_only=True)
            unexpected = model.load_state_dict(state, strict=False).unexpected_keys
            if unexpected:
                raise ValueError(f"Checkpoint task-head keys do not match: {unexpected}")
    model.to(target).eval()
    rows = _records(root / "training_history.csv")
    history = [{key: (None if value == "" else (int(value) if key in {"epoch", "train_examples", "validation_examples", "optimizer_steps"} else float(value)))
                for key, value in row.items()} for row in rows]
    return TrainingRun(model, tokenizer, cfg, history, metadata["best_epoch"], metadata["best_validation"], root, metadata)


def _normalize_id(value: Any) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        raise ValueError("Prediction IDs must not be missing")
    result = str(value).strip().strip('"').strip()
    if not result:
        raise ValueError("Prediction IDs must not be empty")
    return result


@torch.no_grad()
def predict_submit(
    run: TrainingRun, test_frame: Any, filename: str | Path, *,
    probability_filename: Optional[str | Path] = None, id_column: str = "id",
    review_column: Optional[str] = None, sample_submission: Any = None,
) -> PredictionResult:
    """Export hard labels by default, and positive-class probabilities separately.

    Without a template, preserve test-frame order. With a template, validate a
    one-to-one ID match and reorder both exports to the template's ID order.
    """
    rows = _records(test_frame)
    if any(id_column not in row for row in rows):
        raise ValueError(f"Missing prediction ID column {id_column!r}")
    ids = [_normalize_id(row[id_column]) for row in rows]
    if len(ids) != len(set(ids)):
        raise ValueError("Prediction IDs must be unique")
    dataset = ReviewDataset(rows, review_column or run.config.review_column, label_column=None)
    loader = make_data_loader(dataset, run.tokenizer, run.config, training=False)
    target = next(run.model.parameters()).device
    run.model.eval()
    hard_labels, positive = [], []
    for batch in loader:
        with _autocast(target, run.config.amp):
            logits = _logits(run.model, batch, target)
        hard_labels.extend(logits.argmax(dim=-1).cpu().tolist())
        positive.extend(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().tolist())
    if sample_submission is not None:
        template_rows = _records(sample_submission)
        if any(id_column not in row for row in template_rows):
            raise ValueError("Submission template has no ID column")
        template_ids = [_normalize_id(row[id_column]) for row in template_rows]
        if len(template_ids) != len(set(template_ids)) or set(template_ids) != set(ids):
            raise ValueError("Submission template must match prediction IDs one-to-one")
        predictions = dict(zip(ids, zip(hard_labels, positive)))
        hard_labels = [predictions[item][0] for item in template_ids]
        positive = [predictions[item][1] for item in template_ids]
        ids = template_ids
    if not (len(ids) == len(hard_labels) == len(positive)) or not all(math.isfinite(p) and 0 <= p <= 1 for p in positive):
        raise ValueError("Prediction length or probabilities are invalid")
    submission_path = Path(filename)
    probability_path = Path(probability_filename) if probability_filename else submission_path.with_name(submission_path.stem + "_probabilities.csv")
    if submission_path.resolve() == probability_path.resolve():
        raise ValueError("Hard labels and probabilities require separate output paths")
    _write_csv(submission_path, [{id_column: item, "sentiment": label} for item, label in zip(ids, hard_labels)], [id_column, "sentiment"])
    _write_csv(probability_path, [{id_column: item, "positive_probability": p} for item, p in zip(ids, positive)], [id_column, "positive_probability"])
    _write_json(submission_path.with_suffix(".metadata.json"), {
        "model_key": run.config.model_key, "training_scope": run.metadata["training_scope"],
        "prediction_checkpoint": "best_validation", "best_epoch": run.best_epoch,
        "n_examples": len(ids), "submission_values": "hard_labels_0_or_1",
        "probability_values": "softmax_probability_of_label_1", "template_order_used": sample_submission is not None,
        "submission_path": str(submission_path), "probability_path": str(probability_path),
        "kaggle_submitted": False, "kaggle_leaderboard_score": None,
    })
    return PredictionResult(submission_path, probability_path, len(ids), ids, hard_labels, positive)


__all__ = [
    "MODEL_REGISTRY", "ModelSpec", "TrainingConfig", "TrainingRun", "PredictionResult",
    "ReviewDataset", "ReviewCollator", "strip_html", "set_reproducibility",
    "make_data_loader", "build_model_and_tokenizer", "train_model", "evaluate", "load_run", "predict_submit",
]


## 6. 读取已检查的数据并划分

使用第1节已经定位并检查的三个输入。划分默认沿用原Notebook：80%/20%，`random_state=0`，不分层；训练随机种子另外固定为42。

新模型使用子词token数；256个BERT token不能与原Notebook的512个GloVe词视为等长条件，因此这里不是仅替换网络的严格消融。


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

SPLIT_SEED = 0
STRATIFY = False                # 与原Notebook保持相同划分方式。

labeled = read_dataframe(INPUT_FILES["labeledTrainData.tsv"])
test_frame = read_dataframe(INPUT_FILES["testData.tsv"])
sample = read_dataframe(INPUT_FILES["sampleSubmission.csv"])
assert len(labeled) == len(test_frame) == len(sample) == 25000
assert set(labeled["sentiment"].unique()) == {0, 1}
train_indices, validation_indices = train_test_split(
    list(range(len(labeled))), test_size=0.2, random_state=SPLIT_SEED,
    stratify=labeled["sentiment"] if STRATIFY else None,
)
train_frame = labeled.iloc[train_indices].reset_index(drop=True)
validation_frame = labeled.iloc[validation_indices].reset_index(drop=True)
print("train/validation/test:", len(train_frame), len(validation_frame), len(test_frame))


## 7. 只改这里选择模型

默认`SMOKE_ONLY=False`，Run All直接进行完整20,000/5,000训练与验证，默认3轮，并输出完整提交文件。若想先检查新模型的下载、前向/反向与显存，将其设为`True`：只用128条训练样本、64条验证样本跑一轮，输出目录另建；试跑完成后改回`False`，重新运行本节及之后的单元。小样本分数不能填写为完整实验结果。

`deberta_v2_xxlarge` 默认 batch=1、长度128、累积16、梯度检查点。FP16冻结底座的权重粗估约3GB，激活、临时张量和任务层仍要占空间；这些设置需在实际T4会话测峰值，不保证绝不OOM。显存不足先重启释放旧模型，再缩短长度。缩短长度会改变实验条件，必须记录。

XXL的 `xxl_mode="full"` 仅为大显存单卡提供显式入口，默认要求至少40GiB空闲显存，并开启 `allow_xxl_full`；双T4不满足此单卡条件。`head_only`是只训练pooler/classifier的独立基线，不能记为全量或LoRA。


In [ ]:
from datetime import datetime
import gc

MODEL_KEY = "bert_full"      # "bert_full" / "bert_lora" / "deberta_v2_xxlarge"
SMOKE_ONLY = False           # 默认完整训练；True仅做小样本检查，不生成可提交文件。
EPOCHS = 3
DEVICE = "cuda:0"           # 可改cuda:1；每次只使用一张。

cfg = TrainingConfig(
    model_key=MODEL_KEY,
    output_dir=f"/kaggle/working/result/{MODEL_KEY}_{'smoke' if SMOKE_ONLY else 'full'}_{datetime.now():%Y%m%d_%H%M%S}",
    num_epochs=1 if SMOKE_ONLY else EPOCHS,
    device=DEVICE,
    learning_rate=2e-5,
    monitor="accuracy" if SMOKE_ONLY else "auc",
    seed=42,
    amp=True,
    xxl_mode="lora",
    lora_r=8,
    lora_alpha=16,
)
# None按MODEL_REGISTRY采用对应底座默认值。
# 例如需要更长文本时可设cfg.max_length=256；同时留意显存峰值。
# 真实DeBERTa checkpoint固定revision，便于后续重载。
if MODEL_KEY == "deberta_v2_xxlarge":
    cfg.revision = "38f115b79f551ca4091c9caef41940fd1d6312b9"

fit_train = train_frame.iloc[:128].copy() if SMOKE_ONLY else train_frame
fit_validation = validation_frame.iloc[:64].copy() if SMOKE_ONLY else validation_frame
if "run" in globals():
    del run
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(torch.device(DEVICE))
print(MODEL_KEY, MODEL_REGISTRY[MODEL_KEY])
print("小样本试跑" if SMOKE_ONLY else "完整训练", len(fit_train), len(fit_validation))


## 8. 训练、记录与显存峰值

训练同时输出 `training_history.csv`、`training_metadata.json`、`results.json` 与 `best_checkpoint/`。Kaggle分数保持空值，完成提交后再记录。


In [ ]:
try:
    run = train_model(MODEL_KEY, fit_train, fit_validation, config=cfg)
except torch.cuda.OutOfMemoryError:
    print("显存不足：重启会话释放旧模型；XXL保持batch=1，尝试max_length=64，再进行小样本试跑。")
    raise

peak_allocated = torch.cuda.max_memory_allocated(torch.device(DEVICE)) / 2**30
peak_reserved = torch.cuda.max_memory_reserved(torch.device(DEVICE)) / 2**30
print("训练方式:", run.metadata["training_scope"])
print("可训练/总参数:", run.metadata["trainable_parameters"], run.metadata["total_parameters"])
print("最佳轮次:", run.best_epoch, "验证:", run.best_validation)
print(f"PyTorch峰值allocated={peak_allocated:.2f}GiB, reserved={peak_reserved:.2f}GiB")
extra = {
    "smoke_only":SMOKE_ONLY, "split_seed":SPLIT_SEED, "stratify":STRATIFY,
    "peak_allocated_gib":peak_allocated, "peak_reserved_gib":peak_reserved,
    "split_note":"original 80/20, random_state=0; full split stored separately",
    "input_validation":INPUT_REPORT,
}
(run.output_dir / "notebook_settings.json").write_text(json.dumps(extra,indent=2)+"\n",encoding="utf-8")
split_rows = pd.concat([
    pd.DataFrame({"source_row":train_indices,"partition":"train","id":train_frame["id"]}),
    pd.DataFrame({"source_row":validation_indices,"partition":"validation","id":validation_frame["id"]}),
],ignore_index=True)
split_rows.to_csv(run.output_dir / "split_indices.csv",index=False)
display(pd.DataFrame(run.history))


## 9. 用最佳模型导出

试跑模式只预测前16条，并在文件名标记 `smoke`，**不要提交这种小样本文件**。完整模式校验25,000条与提交模板ID一致，再按模板顺序导出。

`submission.csv`的`sentiment`是0/1，保持原Notebook与比赛教程的方式；`*_probabilities.csv`存正类概率，列名为 `positive_probability`，供分析与AUC计算，不能未经调整直接作为比赛的两列模板提交。ROC-AUC使用连续分数能保留排序信息；测试标签未知，不能在本地计算测试AUC。


In [ ]:
prediction_frame = test_frame.iloc[:16].copy() if SMOKE_ONLY else test_frame
prediction_template = None if SMOKE_ONLY else sample
result = predict_submit(
    run, prediction_frame,
    run.output_dir / ("smoke_predictions.csv" if SMOKE_ONLY else "submission.csv"),
    sample_submission=prediction_template,
)
print("预测条数:", result.n_examples)
print("0/1文件:", result.submission_path)
print("正类概率:", result.probability_path)
assert result.n_examples == (16 if SMOKE_ONLY else 25000)
display(pd.read_csv(result.submission_path).head())


## 10. 接下来比较什么

先完整跑BERT全量和BERT LoRA，记录同一验证集的accuracy/AUC、最佳轮次、训练参数量、峰值显存与Kaggle提交分数；同种配置建议重复不同训练种子后再判断差异。DeBERTa用另一个预训练底座和更短的默认长度，单独报告其方法和条件，不能把差异只归因于LoRA。

| 运行 | 验证accuracy | 验证AUC | 最佳轮次 | 显存峰值 | Kaggle AUC |
| --- | --- | --- | --- | --- | --- |
| BERT full | 待运行 | 待运行 | 待运行 | 待实测 | 待提交 |
| BERT LoRA | 待运行 | 待运行 | 待运行 | 待实测 | 待提交 |
| DeBERTa-v2-xxlarge LoRA | 待运行 | 待运行 | 待运行 | 待实测 | 待提交 |

重新打开会话后，可用 `load_run("保存的输出目录", device="cuda:0")`恢复最佳模型。LoRA目录不是完整底座，需要相同checkpoint；重载前释放原`run`，避免同时保留两套模型。

来源：[BERT底座](https://huggingface.co/google-bert/bert-base-uncased)、[DeBERTa-v2-xxlarge模型卡](https://huggingface.co/microsoft/deberta-v2-xxlarge)、[PEFT LoRA](https://huggingface.co/docs/peft/en/package_reference/lora)、[混合精度注意事项](https://huggingface.co/docs/peft/en/developer_guides/troubleshooting)、[Kaggle评分规则](https://www.kaggle.com/competitions/word2vec-nlp-tutorial/overview/evaluation)。
